### Problem: A/B Test Results Analysis & Interpretation
You recently concluded an A/B test for a new checkout flow. You have the raw user-level data showing whether each user clicked 'Purchase' (1) or did not (0). Calculate the conversion rate for each group, run a statistical test (Z-test for proportions) to calculate the p-value and confidence intervals, and interpret if the test is statistically and practically significant given an MDE of 2%.

**Sample Input**:
| user_id | group | converted |
|---|---|---|
| 1 | control | 0 |
| 2 | treatment | 1 |
| 3 | control | 0 |
| 4 | treatment | 0 |

**Sample Output**:
```
Control Conversion: 10.50%
Treatment Conversion: 12.80%
P-Value: 0.012
Confidence Interval: [0.005, 0.041]
Result: Statistically Significant. The treatment increased conversion by 2.3% absolute, which exceeds the 2% MDE.
```

**What you should use:**
- `pandas` to group by variant and calculate conversion rates and counts.
- `statsmodels.stats.proportion.proportions_ztest` to calculate the z-stat and p-value.
- `statsmodels.stats.proportion.proportion_confint` to calculate the 95% confidence intervals for each group.

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Setting up mock dataframe
np.random.seed(42)
control_data = pd.DataFrame({'user_id': range(1, 5001), 'group': 'control', 'converted': np.random.binomial(1, 0.10, 5000)})
treatment_data = pd.DataFrame({'user_id': range(5001, 10001), 'group': 'treatment', 'converted': np.random.binomial(1, 0.125, 5000)})
df = pd.concat([control_data, treatment_data])

df.head()

,user_id,group,converted
0,1,control,0
1,2,control,1
2,3,control,0
3,4,control,0
4,5,control,0


---
### Optimized Solution

In [2]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# 1. Calculate Aggregates
summary = df.groupby('group')['converted'].agg(['sum', 'count']).rename(columns={'sum': 'successes', 'count': 'trials'})
summary['conversion_rate'] = summary['successes'] / summary['trials']

control_cr = summary.loc['control', 'conversion_rate']
treatment_cr = summary.loc['treatment', 'conversion_rate']
absolute_diff = treatment_cr - control_cr

print(f"Control Conversion: {control_cr:.2%}")
print(f"Treatment Conversion: {treatment_cr:.2%}")

# 2. Calculate Z-test (P-value)
successes = summary['successes'].values # [control_successes, treatment_successes]
trials = summary['trials'].values       # [control_trials, treatment_trials]

# Order matters: we usually test (Treatment vs Control) but proportions_ztest tests if prop[0] == prop[1]. 
# It will return a negative z-stat if treatment > control (assuming control is index 0).
# The two-sided p-value is unaffected by order.
z_stat, p_value = proportions_ztest(successes, trials, alternative='two-sided')

print(f"P-Value: {p_value:.3f}")

# 3. Business Interpretation
mde = 0.02 # 2% Absolute
alpha = 0.05

if p_value < alpha:
    if absolute_diff >= mde:
        print(f"Result: Statistically and Practically Significant. The treatment increased conversion by {absolute_diff:.2%} absolute, which exceeds the {mde:.2%} MDE.")
    else:
        print(f"Result: Statistically Significant BUT Practically Insignificant. The {absolute_diff:.2%} increase is less than the {mde:.2%} MDE required to launch.")
else:
    print("Result: Not Statistically Significant. Fail to reject the null hypothesis.")

Control Conversion: 9.58%
Treatment Conversion: 11.76%
P-Value: 0.000
Result: Statistically and Practically Significant. The treatment increased conversion by 2.18% absolute, which exceeds the 2.00% MDE.


---
### Concepts Explained

**1. Why Z-test for Proportions?**
*   When dealing with click-through rates or conversion rates (Yes/No categorical data), we use the **Z-test for Two Proportions**. Because our sample sizes are large (usually > thousands), the sampling distribution of the proportions approaches a normal distribution (CLT), making the Z-test appropriate.

**2. Two-Sided vs One-Sided Tests**
*   We used `alternative='two-sided'`. A one-sided test only checks if Treatment is *better* than Control. A two-sided test checks if it is *better OR worse*. Data Science best practice mandates two-sided tests because you must detect if a feature actively harms the business, not just if it helps.

**3. Practical Significance (The PM check)**
*   Notice the final IF statement. Even if $p < 0.05$, if the absolute difference is only 0.5% and the business needed a 2% (MDE) return to cover engineering costs, the test is a statistical success but a business failure. We do not launch it.